SECTION 1 — IMPORTS + LOAD ARTIFACTS

In [41]:
from pathlib import Path
import json
import math
import re
from collections import Counter


# ============================================================
# PROJECT PATHS
# ============================================================

ROOT = Path(r"C:\amrita_uni\s6\NLP\project\Rubric-based-evaluation-of-PL-SQL-code\Rubric-based-evaluation-of-PL-SQL-code")

SCHEMA_ARTIFACT_DIR = ROOT / "artifacts" / "schema_output"

ARTIFACT_DIR = ROOT / "artifacts" / "rag_output"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOAD SCHEMA ARTIFACTS
# ============================================================

schema = json.loads(
    (SCHEMA_ARTIFACT_DIR / "schema.json")
    .read_text(encoding="utf-8")
)

prompt_decomposition = json.loads(
    (SCHEMA_ARTIFACT_DIR / "prompt_decomposition.json")
    .read_text(encoding="utf-8")
)

schema_confidence = json.loads(
    (SCHEMA_ARTIFACT_DIR / "schema_confidence.json")
    .read_text(encoding="utf-8")
)

print("Schema artifacts loaded.")

Schema artifacts loaded.


SECTION 2 — LOAD SEMANTIC DATASET

In [42]:
# ============================================================
# LOAD ENRICHED SEMANTIC DATASET
# ============================================================

SEMANTIC_DATASET_PATH = (
    ROOT
    / "plsql_mutation_semantic_dataset_v2.jsonl"
)


def load_semantic_mutation_dataset() -> list[dict]:

    documents = []

    with open(
        SEMANTIC_DATASET_PATH,
        "r",
        encoding="utf-8"
    ) as handle:

        for line in handle:

            line = line.strip()

            if not line:
                continue

            documents.append(json.loads(line))

    return documents


mutation_documents = (
    load_semantic_mutation_dataset()
)

print("Semantic mutation documents:",
      len(mutation_documents))

Semantic mutation documents: 85


SECTION 3 — TOKENIZATION UTILITIES

In [43]:
def tokenize(text: str) -> list[str]:

    return re.findall(
        r"[a-zA-Z_][a-zA-Z0-9_]+",
        text.lower()
    )


def count_terms(tokens: list[str]) -> dict:

    counts = {}

    for token in tokens:
        counts[token] = counts.get(token, 0) + 1

    return counts


def cosine_similarity(left: dict, right: dict) -> float:

    if not left or not right:
        return 0.0

    numerator = sum(
        left.get(token, 0.0)
        *
        right.get(token, 0.0)
        for token in left
    )

    left_norm = math.sqrt(
        sum(v * v for v in left.values())
    )

    right_norm = math.sqrt(
        sum(v * v for v in right.values())
    )

    if not left_norm or not right_norm:
        return 0.0

    return numerator / (left_norm * right_norm)

SECTION 4 — BUILD PROCEDURAL RETRIEVAL QUERY

In [44]:
def build_semantic_retrieval_query(
    schema: dict,
    decomposition: dict
) -> dict:
    """
    Build generalized procedural retrieval query.
    """

    procedural_patterns = (
        decomposition["procedural_patterns"]
    )

    risk_patterns = (
        decomposition["risk_patterns"]
    )

    behavioral_expectations = (
        decomposition["behavioral_expectations"]
    )

    entity_names = [

        entity["name"]

        for entity
        in schema.get("entities", [])
    ]

    retrieval_text = "\n".join([

        json.dumps(procedural_patterns),

        json.dumps(risk_patterns),

        json.dumps(behavioral_expectations),

        json.dumps(entity_names)
    ])

    return {

        "procedural_patterns":
            procedural_patterns,

        "risk_patterns":
            risk_patterns,

        "behavioral_expectations":
            behavioral_expectations,

        "entity_names":
            entity_names,

        "retrieval_text":
            retrieval_text
    }


retrieval_query = (
    build_semantic_retrieval_query(
        schema,
        prompt_decomposition
    )
)

print(json.dumps(
    retrieval_query,
    indent=2
))

{
  "procedural_patterns": [
    "boundary_validation",
    "cross_table_dependency",
    "transaction_sensitive_operation"
  ],
  "risk_patterns": [
    "threshold_logic_failure"
  ],
  "behavioral_expectations": [
    "Boundary equality cases must be tested.",
    "Cross-table updates should preserve consistency.",
    "Exact balance withdrawals should succeed.",
    "Negative withdrawal amounts should fail.",
    "Operation should reject insufficient balance.",
    "Partial failures should rollback safely.",
    "Threshold minus one cases should behave correctly.",
    "Threshold plus one cases should behave correctly."
  ],
  "entity_names": [
    "ACCOUNTS",
    "BRANCHES",
    "CUSTOMER"
  ],
  "retrieval_text": "[\"boundary_validation\", \"cross_table_dependency\", \"transaction_sensitive_operation\"]\n[\"threshold_logic_failure\"]\n[\"Boundary equality cases must be tested.\", \"Cross-table updates should preserve consistency.\", \"Exact balance withdrawals should succeed.\", \

SECTION 5 — SEMANTIC DOCUMENT FEATURE EXTRACTION

In [45]:
def document_feature_text(doc: dict) -> str:
    """
    Convert semantic mutation document
    into retrieval text.
    """

    return " ".join([

        doc.get("mutation_name", ""),

        doc.get("category", ""),

        doc.get("semantic_category", ""),

        doc.get("procedural_risk", ""),

        " ".join(
            doc.get("testcase_strategy", [])
        ),

        " ".join(
            doc.get("edge_case_family", [])
        ),

        doc.get("behavioral_focus", ""),

        doc.get("why_it_matters", ""),

        doc.get(
            "expected_observable_effect",
            ""
        ),

        " ".join(doc.get("tags", []))
    ])

SECTION 6 — SEMANTIC MUTATION SCORING

In [46]:
SEMANTIC_CATEGORY_WEIGHTS = {

    "boundary_validation": 2.0,

    "transaction_sensitive_operation": 2.5,

    "cross_table_dependency": 2.5,

    "exception_sensitive_operation": 1.8,

    "null_sensitive_operation": 1.5
}


def score_semantic_mutations(
    documents: list[dict],
    retrieval_query: dict
) -> list[dict]:
    """
    Semantic procedural retrieval scoring.
    """

    query_tokens = tokenize(
        retrieval_query["retrieval_text"]
    )

    query_vector = count_terms(query_tokens)

    ranked = []

    for doc in documents:

        doc_text = document_feature_text(doc)

        doc_tokens = tokenize(doc_text)

        doc_vector = count_terms(doc_tokens)

        lexical_score = cosine_similarity(
            query_vector,
            doc_vector
        )

        semantic_category = (
            doc.get(
                "semantic_category",
                ""
            )
        )

        semantic_score = (
            SEMANTIC_CATEGORY_WEIGHTS.get(
                semantic_category,
                0.5
            )
        )

        testcase_bonus = (
            0.2
            *
            len(doc.get(
                "testcase_strategy",
                []
            ))
        )

        edge_case_bonus = (
            0.15
            *
            len(doc.get(
                "edge_case_family",
                []
            ))
        )

        total_score = (

            lexical_score

            +

            semantic_score

            +

            testcase_bonus

            +

            edge_case_bonus
        )

        ranked.append({

            "id":
                doc.get("id"),

            "mutation_name":
                doc.get("mutation_name"),

            "semantic_category":
                semantic_category,

            "procedural_risk":
                doc.get(
                    "procedural_risk",
                    ""
                ),

            "testcase_strategy":
                doc.get(
                    "testcase_strategy",
                    []
                ),

            "edge_case_family":
                doc.get(
                    "edge_case_family",
                    []
                ),

            "score":
                round(total_score, 4),

            "why_it_matters":
                doc.get(
                    "why_it_matters",
                    ""
                )
        })

    return sorted(
        ranked,
        key=lambda x: x["score"],
        reverse=True
    )

SECTION 7 — DIVERSE PROCEDURAL RETRIEVAL

In [47]:
def select_diverse_semantic_mutations(
    ranked_mutations: list[dict],
    top_k: int = 10
) -> list[dict]:
    """
    Ensure semantic diversity.
    """

    selected = []

    semantic_counts = Counter()

    for mutation in ranked_mutations:

        category = (
            mutation["semantic_category"]
        )

        # ================================================
        # PREVENT CATEGORY DOMINATION
        # ================================================

        if semantic_counts[category] >= 3:
            continue

        selected.append(mutation)

        semantic_counts[category] += 1

        if len(selected) >= top_k:
            break

    return selected


ranked_mutations = (
    score_semantic_mutations(
        mutation_documents,
        retrieval_query
    )
)

retrieved_mutations = (
    select_diverse_semantic_mutations(
        ranked_mutations,
        top_k=10
    )
)

print("\nTop Retrieved Semantic Mutations:\n")

for item in retrieved_mutations:

    print(
        item["id"],
        "|",
        item["semantic_category"],
        "|",
        round(item["score"], 3)
    )


Top Retrieved Semantic Mutations:

MUT-044 | transaction_sensitive_operation | 3.268
MUT-045 | transaction_sensitive_operation | 3.253
MUT-082 | transaction_sensitive_operation | 3.253
MUT-074 | boundary_validation | 3.196
MUT-073 | boundary_validation | 3.192
MUT-009 | boundary_validation | 3.147
MUT-020 | cross_table_dependency | 3.046
MUT-017 | cross_table_dependency | 3.036
MUT-021 | cross_table_dependency | 3.017
MUT-050 | exception_sensitive_operation | 2.632


SECTION 8 — SEMANTIC COVERAGE EVALUATION

In [48]:
SEMANTIC_ALIGNMENT = {

    "transaction_sensitive_operation":
        "transaction_sensitive_operation",

    "boundary_validation":
        "boundary_validation",

    "cross_table_dependency":
        "cross_table_dependency",

    "exception_sensitive_operation":
        "exception_sensitive_operation",

    "null_sensitive_operation":
        "null_sensitive_operation"
}


def evaluate_semantic_rag_quality(
    retrieved_mutations: list[dict],
    retrieval_query: dict
) -> dict:
    """
    Evaluate semantic procedural coverage.
    """

    retrieved_categories = {

        mutation["semantic_category"]

        for mutation
        in retrieved_mutations
    }

    required_patterns = (
        retrieval_query["procedural_patterns"]
    )

    semantic_hits = {}

    covered_patterns = set()

    for pattern in required_patterns:

        aligned = (
            SEMANTIC_ALIGNMENT.get(
                pattern
            )
        )

        hit = aligned in retrieved_categories

        semantic_hits[pattern] = hit

        if hit:
            covered_patterns.add(pattern)

    coverage_score = (

        len(covered_patterns)

        /

        max(1, len(required_patterns))
    )

    missing_patterns = (

        set(required_patterns)

        -

        covered_patterns
    )

    return {

        "coverage_score":
            round(coverage_score, 3),

        "covered_patterns":
            sorted(covered_patterns),

        "missing_patterns":
            sorted(missing_patterns),

        "semantic_hits":
            semantic_hits,

        "retrieved_categories":
            sorted(retrieved_categories),

        "retrieved_count":
            len(retrieved_mutations)
    }


rag_quality = (
    evaluate_semantic_rag_quality(
        retrieved_mutations,
        retrieval_query
    )
)

print(json.dumps(
    rag_quality,
    indent=2
))

{
  "coverage_score": 1.0,
  "covered_patterns": [
    "boundary_validation",
    "cross_table_dependency",
    "transaction_sensitive_operation"
  ],
  "missing_patterns": [],
  "semantic_hits": {
    "boundary_validation": true,
    "cross_table_dependency": true,
    "transaction_sensitive_operation": true
  },
  "retrieved_categories": [
    "boundary_validation",
    "cross_table_dependency",
    "exception_sensitive_operation",
    "transaction_sensitive_operation"
  ],
  "retrieved_count": 10
}


SECTION 9 — EXPORT RAG ARTIFACTS

In [49]:
(ARTIFACT_DIR / "retrieved_mutations.json").write_text(
    json.dumps(
        retrieved_mutations,
        indent=2
    ),
    encoding="utf-8"
)

(ARTIFACT_DIR / "rag_quality.json").write_text(
    json.dumps(
        rag_quality,
        indent=2
    ),
    encoding="utf-8"
)

(ARTIFACT_DIR / "retrieval_query.json").write_text(
    json.dumps(
        retrieval_query,
        indent=2
    ),
    encoding="utf-8"
)

print("RAG artifacts exported.")

RAG artifacts exported.


SECTION 10 — ASSERTION TESTS

In [50]:
assert len(mutation_documents) >= 50

assert (
    rag_quality["coverage_score"]
    >= 0.66
)

assert (
    len(retrieved_mutations)
    >= 7
)

assert (
    rag_quality["retrieved_count"]
    ==
    len(retrieved_mutations)
)

assert (
    len(
        rag_quality["covered_patterns"]
    )
    >= 2
)

print("Semantic RAG notebook tests passed.")

Semantic RAG notebook tests passed.
